# IAN 620 · Module 9 Assignment
## Agentic RAG with LangChain + LangGraph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ali2449/IAN-620/blob/main/IAN_620_Module_9_Assignment_Agentic_RAG_Starter.ipynb)

This is a **starter notebook**, not a completed solution.

You are given the environment setup, dataset loading, and a few small syntax examples. You will build the actual RAG and Agentic RAG workflow.

### Main idea

**Long documents → chunking → embeddings → Chroma → tools → state → decisions → LangGraph**

You should be able to explain every part of the workflow you submit.

## What is already provided?

Provided for you:

- library installation
- imports
- dataset URL and loading
- a very small LangChain syntax example
- local embedding-model setup
- local Qwen model setup
- a tiny unrelated LangGraph syntax example

You must complete:

- converting the assignment dataset into LangChain Documents
- comparing two chunking settings
- choosing the final chunking strategy
- creating Chroma and the retriever
- creating the knowledge-base search tool
- completing the order lookup tool
- defining useful agent state
- routing
- retrieval
- evidence checking
- query rewriting
- generation
- LangGraph edges and conditional routing
- testing and reflection

## 0. Install libraries

In [ ]:
%pip install -q -U \
    langchain \
    langgraph \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    chromadb

## 1. Imports

In [ ]:
import re
import operator
from typing import TypedDict, Annotated

import pandas as pd

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 2. Load the assignment dataset

Each row is a **long source document**.

You should not treat each row as the final retrieval unit. The assignment requires chunking.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ali2449/IAN-620/main/IAN_620_Module_9_Agentic_RAG_Knowledge_Base_Large.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", list(df.columns))

df.head(3)

### Inspect the document lengths

In [ ]:
df["word_count"] = df["document_text"].str.split().str.len()

print("Average words:", round(df["word_count"].mean(), 1))
print("Minimum words:", df["word_count"].min())
print("Maximum words:", df["word_count"].max())

df[["document_id", "title", "category", "word_count"]].head(10)

## 3. Very small LangChain example

This example is **not the assignment solution**. It only shows the syntax of a LangChain `Document` and a text splitter.

In [ ]:
example_doc = Document(
    page_content=(
        "A fictional support policy explains that standard items can be returned "
        "within a stated period. Electronics may have a different return period. "
        "A support agent should check the correct policy before answering."
    ),
    metadata={"source": "example_policy"}
)

example_splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20
)

example_chunks = example_splitter.split_documents([example_doc])

for chunk in example_chunks:
    print(chunk.page_content)
    print(chunk.metadata)
    print("---")

## 4. Assignment Task — create LangChain Documents

Create one LangChain `Document` for each CSV row.

Use:

- `document_text` as `page_content`
- useful columns such as `document_id`, `title`, `category`, `region`, and `last_updated` as metadata

Do not copy every column unless you think it will help retrieval or filtering.

<details>
<summary><strong>Hint</strong></summary>

Loop through the DataFrame rows. For each row, create a `Document` where:

- `page_content` comes from `document_text`
- `metadata` is a dictionary with useful fields such as `document_id`, `title`, `category`, and `region`

Then append each document to the `documents` list.

</details>

In [ ]:
# TODO: build the documents list

documents = []

# Write your loop here.
# Hint: create Document(page_content=..., metadata={...})
# and append it to documents.

print("Documents created:", len(documents))

## 5. Assignment Task — compare two chunking strategies

Use `RecursiveCharacterTextSplitter`.

Create **two different** chunking configurations. Do not simply use the values from the small example above.

For each strategy, report:

- chunk size
- overlap
- number of chunks created

Then inspect at least two chunks from each strategy.

<details>
<summary><strong>Hint</strong></summary>

Create two separate `RecursiveCharacterTextSplitter` objects with different values for:

- `chunk_size`
- `chunk_overlap`

Then call `.split_documents(documents)` for each splitter.

Try to make the two settings meaningfully different so you can compare them.

</details>

In [ ]:
# TODO: choose two different settings

CHUNK_SIZE_A = None
CHUNK_OVERLAP_A = None

CHUNK_SIZE_B = None
CHUNK_OVERLAP_B = None

# TODO:
# 1. create splitter_a
# 2. create splitter_b
# 3. create chunks_a
# 4. create chunks_b

# print("Strategy A chunks:", ...)
# print("Strategy B chunks:", ...)

### Inspect your chunks

Use this space to display sample chunks from both strategies.

In [ ]:
# TODO: print at least two chunks from each strategy



### Choose your final chunking strategy

Explain briefly:

- Which strategy did you choose?
- Why?
- Did one strategy preserve context better?
- Did one strategy create chunks that were too broad or too narrow?

**Your answer:**



In [ ]:
# TODO: assign your chosen chunk list to this variable

chunks = None

# Example:
# chunks = chunks_a

print("Final chunks:", 0 if chunks is None else len(chunks))

## 6. Local embeddings

The embedding model is provided. You will use it when creating Chroma.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready.")

## 7. Assignment Task — create Chroma and a retriever

Create a Chroma collection from your final chunks.

Then create a retriever that returns a small number of relevant chunks.

Choose the value of `k` yourself.

<details>
<summary><strong>Hint</strong></summary>

You will need:

1. `Chroma.from_documents(...)`
2. your final `chunks`
3. the provided `embedding_model`
4. `.as_retriever(...)`

For the retriever, use a small `k` such as 3, 4, or 5.

</details>

In [ ]:
# TODO: create the vector database

vector_db = None

# TODO: create the retriever
retriever = None

# When finished, print a short message confirming that retrieval is ready.

## 8. Assignment Task — test retrieval before building an agent

Test at least two questions directly with the retriever.

For each result, display useful metadata and part of the retrieved text.

Do this **before** creating the agent.

<details>
<summary><strong>Hint</strong></summary>

A retriever can be called with:

`retriever.invoke(question)`

The result is a list of LangChain `Document` objects.

For each result, inspect both:

- `doc.page_content`
- `doc.metadata`

</details>

In [ ]:
test_question_1 = "Write your first test question here"
test_question_2 = "Write your second test question here"

# TODO:
# retrieve results for each question
# print title, category, region, and part of page_content


## 9. Small LangChain tool example

This unrelated example only shows the syntax of the `@tool` decorator.

In [ ]:
@tool
def count_words(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())

print(count_words.invoke("LangChain tools wrap normal Python functions."))

## 10. Assignment Task — create the knowledge-base tool

Create a tool named `search_knowledge_base`.

It should:

1. accept a query,
2. call your retriever,
3. return retrieved text,
4. include useful source metadata.

Do not return only the answer. The agent needs the evidence.

<details>
<summary><strong>Hint</strong></summary>

Inside the tool:

1. call the retriever with the query,
2. loop through the returned documents,
3. combine source metadata and chunk text into readable strings,
4. return the combined evidence.

The tool should return evidence, not a final answer.

</details>

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the support knowledge base."""

    # TODO: retrieve relevant chunks
    # TODO: format the chunks with useful source information
    # TODO: return the formatted evidence

    raise NotImplementedError("Complete search_knowledge_base")

## 11. Second source — sample order data

The dictionary is provided so you can focus on tool design.

Add **at least one additional order** of your own.

In [ ]:
orders = {
    "1042": {
        "status": "delivered",
        "days_since_delivery": 20,
        "product": "laptop",
        "region": "US - East",
    },
    "2088": {
        "status": "processing",
        "days_since_delivery": None,
        "product": "camera",
        "region": "US - West",
    },
    "3310": {
        "status": "delivered",
        "days_since_delivery": 5,
        "product": "headphones",
        "region": "US - Central",
    },

    # TODO: add at least one more order
}

## 12. Assignment Task — create the order lookup tool

Complete the tool below.

If the order exists, return useful order information.

If it does not exist, return a clear message.

<details>
<summary><strong>Hint</strong></summary>

Use:

`orders.get(order_id)`

If it returns `None`, return a clear “not found” message.

Otherwise, format the order fields into a readable string.

</details>

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its order number."""

    # TODO: look up order_id in the orders dictionary
    # TODO: format the result
    # TODO: handle an order that does not exist

    raise NotImplementedError("Complete lookup_order")

## 13. Local Qwen model

The model-loading code is provided because model setup is not the main goal of this assignment.

No paid API key is required.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=180,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=text_generator)

print("Local LLM ready.")

## 14. Tiny LangGraph syntax example

This example has nothing to do with the assignment data.

Its only purpose is to show the basic syntax:

1. define state,
2. create nodes,
3. add nodes,
4. add edges,
5. compile,
6. invoke.

In [ ]:
class DemoState(TypedDict):
    message: str


def add_label(state: DemoState):
    return {"message": "Processed: " + state["message"]}


demo_builder = StateGraph(DemoState)

demo_builder.add_node("add_label", add_label)
demo_builder.add_edge(START, "add_label")
demo_builder.add_edge("add_label", END)

demo_graph = demo_builder.compile()

demo_graph.invoke({"message": "hello"})

## 15. Assignment Task — design your agent state

Your state should contain the information that must survive between graph steps.

Possible fields include:

- original question
- current retrieval query
- selected route
- order number
- retrieved evidence
- order result
- whether evidence is sufficient
- retry count
- final answer
- visible execution trace

You do not need to use every field listed above.

<details>
<summary><strong>Hint</strong></summary>

Ask yourself: “Which values must survive from one node to another?”

A minimal state might need fields for:

- question
- current query
- route
- retrieved evidence
- order result
- evidence status
- retry count
- final answer

You do not need every possible field.

</details>

In [ ]:
class AgentState(TypedDict):
    # TODO: define the state fields your workflow needs
    pass

## 16. Assignment Task — routing node

Create a node that decides what information is needed.

At minimum, your workflow should distinguish between:

- a question that only needs the knowledge base
- a question that also needs an order lookup

You may add another simple route if you want.

Keep deterministic decisions in normal Python where possible.

<details>
<summary><strong>Hint</strong></summary>

A simple deterministic router can inspect the question.

For example:

- if it contains an order number, the workflow may need both retrieval and order lookup,
- otherwise, it may only need the knowledge base.

You may use a regular expression or simple keyword checks.

</details>

In [ ]:
def route_node(state: AgentState):
    # TODO:
    # inspect the question
    # decide which source(s) are needed
    # return updates to state

    raise NotImplementedError("Complete route_node")

## 17. Assignment Task — retrieval node

<details>
<summary><strong>Hint</strong></summary>

Use the current retrieval query from state and call:

`search_knowledge_base.invoke(...)`

Then return a dictionary containing the updated evidence field.

</details>

In [ ]:
def retrieve_node(state: AgentState):
    # TODO:
    # call search_knowledge_base
    # save the evidence in state

    raise NotImplementedError("Complete retrieve_node")

## 18. Assignment Task — order node

<details>
<summary><strong>Hint</strong></summary>

Read the order number from state, call `lookup_order`, then return the result back into state.

This node should not decide policy. Its job is only to retrieve structured order information.

</details>

In [ ]:
def order_node(state: AgentState):
    # TODO:
    # call lookup_order when the selected route needs it
    # save the result in state

    raise NotImplementedError("Complete order_node")

## 19. Assignment Task — evidence check

Create a simple evidence check using at least **two signals**.

Examples:

- amount of retrieved text
- question keywords found in retrieved chunks
- correct category
- correct region
- order found / not found

The evidence check should produce a simple result such as `True` or `False`.

<details>
<summary><strong>Hint</strong></summary>

Use two simple signals.

For example:

- retrieved evidence is longer than a minimum length,
- and the evidence contains an important keyword from the question,
- or an order lookup succeeded when an order was required.

Combine the signals into one `True` / `False` result.

</details>

In [ ]:
def grade_node(state: AgentState):
    # TODO:
    # use at least two signals
    # decide whether the evidence is sufficient
    # save that decision in state

    raise NotImplementedError("Complete grade_node")

## 20. Assignment Task — query rewriting

If evidence is weak, create a better retrieval query.

Do not simply repeat the exact same query.

Think about:

- important product terms
- policy keywords
- region
- category
- user intent

Also increase the retry count.

<details>
<summary><strong>Hint</strong></summary>

Do not replace the user’s intent.

Start with the original question and add terms that make the retrieval more specific, such as:

- policy
- eligibility
- return
- shipping
- region
- product type

Also increment the retry counter.

</details>

In [ ]:
def rewrite_node(state: AgentState):
    # TODO:
    # create a more useful retrieval query
    # increase the retry count

    raise NotImplementedError("Complete rewrite_node")

## 21. Assignment Task — answer generation

Create a prompt that gives Qwen:

- the original user question
- retrieved evidence
- order information when available

Your prompt should tell the model:

- use only the provided evidence,
- do not invent missing facts,
- say when evidence is insufficient,
- keep the answer concise.

<details>
<summary><strong>Hint</strong></summary>

Build one prompt that includes:

- the user question,
- retrieved evidence,
- order information if available,
- clear grounding instructions.

Then call:

`llm.invoke(prompt)`

Return only the final answer into state.

</details>

In [ ]:
def generate_node(state: AgentState):
    # TODO:
    # build a grounded prompt
    # call llm.invoke(...)
    # save the final answer in state

    raise NotImplementedError("Complete generate_node")

## 22. Assignment Task — stopping and conditional routing

Set a maximum number of retrieval rewrites.

Then create the conditional functions needed for your graph.

Your workflow should not loop forever.

<details>
<summary><strong>Hint</strong></summary>

Think of the grading decision like this:

- evidence is good → generate
- evidence is weak and retries remain → rewrite
- retry limit reached → generate a cautious answer or stop

Keep `MAX_REWRITES` small.

</details>

In [ ]:
MAX_REWRITES = None   # TODO: choose a small number


def after_retrieval(state: AgentState):
    # TODO: decide whether to go to order lookup or evidence grading
    raise NotImplementedError("Complete after_retrieval")


def after_grade(state: AgentState):
    # TODO:
    # if evidence is good -> generate
    # if evidence is weak but retries remain -> rewrite
    # if retry limit is reached -> generate a cautious answer or stop
    raise NotImplementedError("Complete after_grade")

## 23. Assignment Task — build the LangGraph

Use your functions from the previous sections.

Your graph should include the main idea below, but you must write the graph code yourself:

**START → route → retrieve → optional order lookup → grade → generate OR rewrite → retrieve again**

Remember:

- use `add_node(...)`
- use normal edges where the next step is fixed
- use conditional edges where the next step depends on state
- finish with `END`
- compile the graph

<details>
<summary><strong>Hint</strong></summary>

A typical build order is:

1. create `StateGraph(AgentState)`
2. add each node with `add_node()`
3. connect `START`
4. add fixed edges
5. add conditional edges for routing and grading
6. connect the final node to `END`
7. call `.compile()`

The rewrite node should loop back to retrieval.

</details>

In [ ]:
# TODO: build and compile your graph

builder = StateGraph(AgentState)

# TODO: add nodes

# TODO: add START edge

# TODO: add normal and conditional edges

# TODO: connect the final answer to END

# agent = builder.compile()


## 24. Assignment Task — run the agent

Create an initial state and invoke your compiled graph.

Display:

- question
- route or visible trace
- final answer

Do not display hidden chain-of-thought.

<details>
<summary><strong>Hint</strong></summary>

Create an initial-state dictionary containing values for every required state field.

Then call:

`agent.invoke(initial_state)`

Display only visible workflow information such as route, retries, sources, and final answer.

</details>

In [ ]:
def run_agent(question):
    # TODO: create the initial state
    # TODO: invoke your graph
    # TODO: print a readable summary
    # TODO: return the result

    raise NotImplementedError("Complete run_agent")

## 25. Final testing

Test at least **four questions**:

1. a knowledge-base-only question,
2. a question with an order number,
3. a question using the new order you added,
4. a question that causes your corrective retrieval / rewrite path to run.

Create your own questions.

<details>
<summary><strong>Hint</strong></summary>

Choose questions that force different paths.

Good tests include:

- a simple policy question,
- a policy + order question,
- a question using the order you added,
- a vague question that benefits from query rewriting.

</details>

In [ ]:
test_questions = [
    # TODO: add at least four questions
]

# TODO: run your agent for each question


## 26. Show retrieval evidence

For at least one test question, display the retrieved chunks and their metadata.

This should make it possible to verify whether the final answer was grounded in the source documents.

<details>
<summary><strong>Hint</strong></summary>

Run the retriever directly for one of your final questions.

Print:

- document id
- title
- category
- region
- a portion of the retrieved chunk

This lets you compare the final answer with the actual evidence.

</details>

In [ ]:
# TODO: show retrieved evidence for one final test question


## 27. Reflection

Answer each question in a few sentences.

### 1. What makes your workflow agentic instead of normal RAG?

Your answer:

### 2. Which two chunking strategies did you compare?

Your answer:

### 3. Which chunking strategy did you choose and why?

Your answer:

### 4. How does your workflow decide when to use the order tool?

Your answer:

### 5. What two signals does your evidence check use?

Your answer:

### 6. What happens when the first retrieval is weak?

Your answer:

### 7. What prevents your workflow from looping forever?

Your answer:

### 8. Give one example where the final answer used information from more than one source.

Your answer:

# Submission checklist

Before submitting:

- [ ] I created LangChain Documents from the CSV.
- [ ] I preserved useful metadata.
- [ ] I compared two chunking strategies.
- [ ] I selected and explained my final chunking strategy.
- [ ] I created local embeddings.
- [ ] I created Chroma and a retriever.
- [ ] I tested retrieval before building the agent.
- [ ] I created a knowledge-base tool.
- [ ] I completed the order tool.
- [ ] I added at least one order.
- [ ] I defined agent state.
- [ ] I implemented routing.
- [ ] I implemented evidence checking with at least two signals.
- [ ] I implemented query rewriting.
- [ ] I added a retry / stop condition.
- [ ] I built the LangGraph myself.
- [ ] I tested at least four questions.
- [ ] I showed retrieval evidence for at least one result.
- [ ] I completed the reflection.

A simple workflow that you understand is better than a complicated workflow copied from somewhere else.